## Capstone Project 

This project demonstrates my ability to build a real-time data lakehouse architecture by integrating structured and semi-structured data from MySQL, MongoDB, and local file systems using PySpark Structured Streaming in a Jupyter Notebook environment. It showcases my implementation of a Lambda architecture with bronze, silver, and gold tables, enabling the ingestion and transformation of streaming fact data alongside static reference data. Dimension tables are constructed from SQL queries, CSV files, and NoSQL collections, emphasizing my understanding of ETL pipelines, foreign key relationships, and data modeling across OLTP and OLAP systems. This project will extend the foundation built in my midterm by incorporating near real-time data processing and layered data refinement to support diagnostic business analysis.

#### Import the Necessary Libraries

In [5]:
import os
import json
import pandas as pd
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col
from delta import configure_spark_with_delta_pip

#### Instantiate Global Variables

In [8]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "host_name": "localhost",
    "port": "3306",
    "db_name": "adventureworks",
    "conn_props": {
        "user": "root",
        "password": "Coronado",    
        "driver": "com.mysql.cj.jdbc.Driver"
    }
}

# --------------------------------------------------------------------------------
# Specify MongoDB Cluster Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "cluster_location": "atlas",
    "user_name": "brw6pe",
    "password": "Coronado",
    "cluster_name": "Lab4Cluster",
    "cluster_subnet": "xgxx9",
    "db_name": "adventureworks",    
    "collection": "",               #will fill this in later per dimension
    "null_column_threshold": 0.5
}

# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'capstone_data')   # renamed from 'lab_data'
data_dir = os.path.join(base_dir, 'adventureworks')     
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

# Example stream sources 
sales_orders_stream_dir = os.path.join(stream_dir, 'sales_orders')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

# Bronze/Silver/Gold Paths for Fact Table Example
sales_orders_output_bronze = os.path.join(database_dir, 'fact_sales_orders', 'bronze')
sales_orders_output_silver = os.path.join(database_dir, 'fact_sales_orders', 'silver')
sales_orders_output_gold = os.path.join(database_dir, 'fact_sales_orders', 'gold')

#### Define Global Functions

In [13]:
# --------------------------------------------------------------------------------
# Create Spark Session (Lab 5 Configuration with Delta 3.3.0 and MySQL support)
# --------------------------------------------------------------------------------
from delta import configure_spark_with_delta_pip

def start_spark(app_name: str = "CapstoneProject") -> SparkSession:
    worker_threads = f"local[{int(os.cpu_count() / 2)}]"
    shuffle_partitions = int(os.cpu_count())

    builder = SparkSession.builder \
        .appName(app_name) \
        .master(worker_threads) \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "2g") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.adaptive.enabled", "false") \
        .config("spark.sql.debug.maxToStringFields", 50) \
        .config("spark.sql.shuffle.partitions", shuffle_partitions) \
        .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
        .config("spark.sql.streaming.schemaInference", "true") \
        .config("spark.sql.warehouse.dir", sql_warehouse_dir) \
        .config("spark.streaming.stopGracefullyOnShutdown", "true") \
        .config("spark.sql.catalogImplementation", "hive") \
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.3.0,mysql:mysql-connector-java:8.0.29") \
        .enableHiveSupport()

    # Apply Delta pip integration
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    return spark

# --------------------------------------------------------------------------------
# Read Table from MySQL
# --------------------------------------------------------------------------------
def read_mysql_table(spark: SparkSession, table_name: str) -> DataFrame:
    url = f"jdbc:mysql://{mysql_args['host_name']}:{mysql_args['port']}/{mysql_args['db_name']}"
    df = spark.read.jdbc(
        url=url,
        table=table_name,
        properties=mysql_args['conn_props']
    )
    return df

# --------------------------------------------------------------------------------
# Read JSON from MongoDB Atlas
# --------------------------------------------------------------------------------
def read_mongodb_collection(spark: SparkSession, collection: str) -> DataFrame:
    mongo_uri = (
        f"mongodb+srv://{mongodb_args['user_name']}:{mongodb_args['password']}"
        f"@{mongodb_args['cluster_name']}.{mongodb_args['cluster_subnet']}.mongodb.net/"
        f"{mongodb_args['db_name']}.{collection}?retryWrites=true&w=majority"
    )
    
    df = spark.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("uri", mongo_uri) \
        .load()
    return df

# --------------------------------------------------------------------------------
# Drop Columns with Mostly Null Values
# --------------------------------------------------------------------------------
def drop_null_columns(df: DataFrame, threshold: float = mongodb_args['null_column_threshold']) -> DataFrame:
    total_rows = df.count()
    null_columns = [col for col in df.columns if df.filter(df[col].isNull()).count() / total_rows >= threshold]
    return df.drop(*null_columns)

# --------------------------------------------------------------------------------
# Write Delta Table to Bronze/Silver/Gold Path
# --------------------------------------------------------------------------------
def write_delta_table(df: DataFrame, path: str, mode: str = "overwrite"):
    df.write.format("delta").mode(mode).save(path)

# --------------------------------------------------------------------------------
# Read Delta Table
# --------------------------------------------------------------------------------
def read_delta_table(spark: SparkSession, path: str) -> DataFrame:
    return spark.read.format("delta").load(path)

# --------------------------------------------------------------------------------
# Start the Session
# --------------------------------------------------------------------------------
spark = start_spark()

25/05/07 11:17:01 WARN Utils: Your hostname, MacBook-Air-6.local resolves to a loopback address: 127.0.0.1; using 192.168.1.34 instead (on interface en0)
25/05/07 11:17:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/emily/.ivy2/cache
The jars for the packages stored in: /Users/emily/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e74a556d-9ceb-4fa8-bd8e-97ba75e50c59;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 64ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|    

:: loading settings :: url = jar:file:/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


25/05/07 11:17:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


#### Extract and Write dim_date from MySQL to Lakehouse

In [16]:
# Define the table name and output path
dim_date_table = "dim_date"
dim_date_path = os.path.join(database_dir, dim_date_table)

# Read the dim_date table from MySQL
dim_date_df = spark.read.jdbc(
    url=f"jdbc:mysql://{mysql_args['host_name']}:{mysql_args['port']}/{mysql_args['db_name']}",
    table=dim_date_table,
    properties=mysql_args["conn_props"]
)

# Write the dataframe to the Lakehouse as a Delta Table
dim_date_df.write.format("delta").mode("overwrite").save(dim_date_path)

# Register the table in the metastore
spark.sql(f"CREATE DATABASE IF NOT EXISTS {dest_database}")
spark.sql(f"USE {dest_database}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dim_date_table}
    USING DELTA
    LOCATION '{dim_date_path}'
""")

# Step 5: Quick check
spark.sql(f"SELECT * FROM {dim_date_table} LIMIT 5").show()

Py4JJavaError: An error occurred while calling o55.jdbc.
: java.lang.ClassNotFoundException: com.mysql.cj.jdbc.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:593)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:526)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationProvider.scal